In [ ]:
# --- Setup: make the `ecp` support package available -----------------
# Colab opens a single notebook and installs nothing, so fetch `ecp` from
# the public repo if it isn't importable yet. On Binder/local it is already
# installed, so this cell is a fast no-op there.
try:
    import ecp  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/ramador09/elementary-computational-physics-binder@main"],
        check=True,
    )


# 3.6 Matrix Functions and the Matrix Exponential

In [ ]:
from ecp.style import header, use_style

use_style()
header(
    volume="Volume III — Eigenvalues and Dynamics",
    number="3.6",
    title="Matrix Functions and the Matrix Exponential",
    blurb="Feed a matrix to a scalar function through its eigenvalues and "
    "the function of a matrix is born — with the exponential as the star, "
    "solving every linear differential equation at once. The famous "
    "hazard comes along: the Taylor series that defines it is, at "
    "moderate norm, among the dubious ways to compute it, and this "
    "notebook measures exactly where definition and algorithm part "
    "company.",
    difficulty="advanced",
    estimate="120–150 min",
)

## Notebook overview

This notebook completes Volume III's arc: eigenvalues were computed
([§3.1](eigenvalues-diagonalization.ipynb)), classified
([§3.2](spectral-theorem.ipynb)–[§3.4](hermitian-unitary-normal.ipynb)),
and stress-tested ([§3.5](schur-jordan-nonnormality.ipynb)); now they
*evaluate functions*. The rule $f(A) = Vf(\Lambda)V^{-1}$ turns any
scalar function into a matrix function, gated here against SciPy's
`expm` and `sqrtm` at $10^{-13}$, with $\det e^{A} = e^{\operatorname{tr}A}$
— the cleanest eigenvalue identity in the subject — at rounding.

The centrepiece is the cautionary ladder from Moler and Van Loan's
*Nineteen Dubious Ways* {cite}`moler2003`: the Taylor series, applied
to one fixed non-normal matrix at three scales, loses **seven digits**
($\lVert A\rVert = 90$), then **25%** ($\lVert A\rVert = 179$), then
returns pure garbage — relative error $10^{22}$ — at
$\lVert A\rVert = 359$, because intermediate terms reach $10^{28}$
while the answer is $10^{-2}$: catastrophic cancellation measured as a
ledger, not asserted as folklore. **Scaling-and-squaring** (written by
hand: halve the norm $s$ times, Taylor safely, square $s$ times) fixes
all three rungs at $10^{-14}$ — the definition and the algorithm are
different objects, which is the volume's closing lesson about *every*
matrix computation.

The exponential then does its day job: $\dot x = Ax$ solved by
$e^{At}x_0$ against `solve_ivp` at $10^{-9}$, $\tfrac{d}{dt}e^{At} =
Ae^{At}$ by central differences, phase portraits animated for a spiral
sink, a saddle and a centre — and the **transient hump**: a stable
non-normal $A$ (both eigenvalues negative) whose $\lVert e^{At}\rVert$
*grows* to 3.4 before decaying, [§3.5](schur-jordan-nonnormality.ipynb)'s
non-normality arriving in dynamics, gated as existing. Square roots
and logarithms round-trip at $10^{-13}$, and the
Baker–Campbell–Hausdorff correction closes the notebook:
$e^{sA}e^{sB} - e^{s(A+B)}$ tracks $\tfrac{s^2}{2}\lVert[A,B]\rVert$
to 0.3% as $s \to 0$ — noncommutativity, measured at its leading
order.

> **How to read a check.** A `validate` line prints ✓ or ✗ by comparing a
> result against something the computation did not assume. A ✗ flags a
> mismatch to investigate, never a verdict on its own.

> **Scope.** Moler and Van Loan {cite}`moler2003`; Higham
> {cite}`higham2008functions` for the modern theory; Golub and
> Van Loan {cite}`golub2013` Chapter 9. The ODE cross-checks use
> `scipy.integrate.solve_ivp` at DOP853 tolerances.

## Theory in brief

### Functions through the spectrum

For diagonalizable $A = V\Lambda V^{-1}$,

```{math}
:label: eq-mf-def
f(A) \;=\; V\,\operatorname{diag}\bigl(f(\lambda_1), \dots,
f(\lambda_n)\bigr)\,V^{-1},
```

consistent with power series and with the Cauchy integral; for the
exponential it gives $e^{A} = Ve^{\Lambda}V^{-1}$ and the identity
$\det e^{A} = \prod e^{\lambda_i} = e^{\operatorname{tr}A}$. The
definition is impeccable; as an *algorithm* it inherits
$\kappa(V)$ — fine for the well-conditioned eigenbases used here,
treacherous near defectiveness ([§3.5](schur-jordan-nonnormality.ipynb)).

### Definition versus algorithm

The series $e^{A} = \sum A^k/k!$ converges for every $A$ — in exact
arithmetic. In floating point its intermediate terms grow like
$\lVert A\rVert^k/k!$ before decaying, and the final sum cancels them
down to $\lVert e^{A}\rVert$, which for stable non-normal $A$ is
*small*: the roundoff floor is $\varepsilon \cdot
\max_k\lVert A^k/k!\rVert$, and once that exceeds the answer the
digits are gone. **Scaling and squaring**,

```{math}
:label: eq-mf-ss
e^{A} \;=\; \bigl(e^{A/2^{s}}\bigr)^{2^{s}},
\qquad \lVert A\rVert/2^{s} \lesssim 1,
```

keeps every intermediate tame — the same sum, restructured, and the
whole difference between a definition and an algorithm.

### Dynamics, humps, and BCH

$e^{At}x_0$ solves $\dot x = Ax$; asymptotic stability is
$\operatorname{Re}\lambda_i < 0$ — but the *transient* obeys
non-normality, not eigenvalues: $\lVert e^{At}\rVert$ can climb far
above 1 before the decay wins ([§3.5](schur-jordan-nonnormality.ipynb)'s
pseudospectra, in motion). And exponentials do not multiply like
scalars:

```{math}
:label: eq-mf-bch
e^{sA}e^{sB} \;=\; e^{s(A+B) + \tfrac{s^2}{2}[A,B] + O(s^3)},
```

with the commutator as the leading correction — the measurable seed
of Lie theory, and of every splitting method in numerical ODEs.

---
## Setup

Data only: the notebook's seeded rng and the Moler-Van Loan cautionary
matrix. Every method here — the spectral rule, the Taylor ladder,
scaling-and-squaring — is built in the exercises.

The Setup below holds this notebook's data and instruments — nothing you
are asked to build. It is collapsed so the building stays yours; expand it
whenever you want the details.

<!-- setup-policy: v2 -->

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from scipy.linalg import expm, sqrtm, logm
from scipy.integrate import solve_ivp

from ecp import animate, validate
from ecp.style import use_style

use_style()
rng = np.random.default_rng(0)  # every random array below comes from this seed

EPS = np.finfo(float).eps

# Moler & Van Loan's cautionary matrix, used at three scales in Ex. 2
A_MV = np.array([[-49.0, 24.0], [-64.0, 31.0]])

## Exercise 1: f(A) through the eigenvalues

**Part a)** Implement {eq}`eq-mf-def` as `f_eig(A, f)` and gate it on
a well-conditioned diagonalizable $5\times5$ matrix: `f_eig(A, exp)`
against `scipy.linalg.expm` and `f_eig(A, sqrt)` against `sqrtm`
(SPD input), both at $10^{-11}$ relative.

**Write this one yourself** — three lines, and the definition of the
whole subject.

**Part b)** Gate the subject's cleanest identity at rounding:
$\det e^{A} = e^{\operatorname{tr}A}$ (the product of
$e^{\lambda_i}$ meets the sum of $\lambda_i$) — and its practical
corollary: $e^{A}$ is *always* invertible, whatever $A$.

In [ ]:
# (solution hidden on the public site)


### Validation 1

In [ ]:
validate.check(
    gap_exp < 1e-11 and gap_sqrt < 1e-11,
    "V f(Lambda) V^-1 matches scipy's expm and sqrtm (Eq. 1)",
    f"{gap_exp:.0e} and {gap_sqrt:.0e} on well-conditioned eigenbases — "
    "the definition, checked against two independent algorithms",
)
validate.below(
    det_gap, 1e-11,
    "and det(e^A) = e^tr(A), the subject's cleanest identity",
    "product of e^lambda meets sum of lambda — so the exponential of "
    "anything is invertible",
)

## Exercise 2: The dubious way, measured on a ladder

{cite}`moler2003` made this matrix famous; three scales of it make the
failure quantitative.

**Part a)** Sum the Taylor series (120 terms, far past numerical
convergence) for $cA_{\text{MV}}$ at $c = 1, 2, 4$, recording the
largest intermediate term. The cancellation ledger: intermediate
$10^{6}, 10^{14}, 10^{28}$ against answers of size
$1.5, 0.5, 0.07$ — the roundoff floor $\varepsilon\cdot\max_k$ crosses
the answer between the rungs.

**Part b)** Gate the ladder (deterministic arithmetic — the same sum
in the same order every run): relative error worse than $10^{-11}$ at
$c = 1$ (seven digits gone), worse than $10^{-2}$ at $c = 2$, worse
than $1$ — nonsense — at $c = 4$.

**Part c)** Write `expm_ss(A)` — {eq}`eq-mf-ss`: scale by $2^{-s}$
until the norm is at most 1, Taylor the tame matrix, square $s$
times — and gate it at $10^{-12}$ against `expm` on **all three
rungs**: same series, restructured, every digit back.

**Part d)** Draw the error against term count for $c = 1$ and
$c = 4$: the good curve converges to its floor; the bad one *stops
improving at the cancellation floor* long before the mathematics is
done — the picture of a definition failing as an algorithm.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 2

In [ ]:
validate.check(
    ladder[1.0][0] > 1e-11 and ladder[2.0][0] > 1e-2
    and ladder[4.0][0] > 1.0,
    "the Taylor ladder fails on schedule: seven digits, then 25%, then "
    "nonsense",
    f"errors {ladder[1.0][0]:.0e}, {ladder[2.0][0]:.0e}, "
    f"{ladder[4.0][0]:.0e} as intermediates reach 1e6, 1e14, 1e28 "
    "against O(1) answers — deterministic cancellation, gated on the "
    "failing side because the same sum runs in the same order every "
    "time",
)
validate.check(
    worst_ss < 1e-12,
    "while scaling-and-squaring repairs all three rungs (Eq. 2)",
    f"worst {worst_ss:.0e} — the identical series with tame "
    "intermediates: algorithms are restructurings, and restructuring "
    "is everything",
)

## Exercise 3: The exponential at its day job

**Part a)** Gate the generator property: for a random $4\times4$
$A$, $\tfrac{d}{dt}e^{At}\big|_{t=1} = Ae^{A}$ by central differences
($h = 10^{-6}$) at $10^{-7}$ relative.

**Part b)** Gate the flow against an independent integrator:
$e^{At}x_0$ at $t = 2$ versus `solve_ivp` (DOP853,
`rtol = 10^{-12}`) at $10^{-9}$ — two completely different methods,
one trajectory.

**Part c)** The transient hump: $A_h = \begin{pmatrix}-1 & 10\\
0 & -1.2\end{pmatrix}$ has both eigenvalues negative, yet
$\lVert e^{A_ht}\rVert_2$ climbs to **3.4** before decaying — gate
the hump's existence ($\max_t > 2$ with all
$\operatorname{Re}\lambda < 0$) and draw it: asymptotic stability is
an eigenvalue statement, transient growth is a non-normality
statement, and confusing them is how stable systems break things
([§3.5](schur-jordan-nonnormality.ipynb), now with $t$ on the axis).

**Part d)** Animate the three canonical phase portraits — spiral
sink, saddle, centre — as flows $e^{At}x_0$ of a ring of initial
conditions: eigenvalues with negative real part spiral in, a real
positive/negative pair stretches along one eigenvector while
squeezing the other, and a purely imaginary pair rotates forever
(with $\det e^{At} = e^{t\operatorname{tr}A} = 1$: areas preserved).

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 3

In [ ]:
validate.check(
    gap_gen < 1e-7,
    "the exponential generates its own derivative: d/dt e^At = A e^At",
    f"{gap_gen:.0e} by central differences — the property that makes "
    "e^At the solution of x' = Ax",
)
validate.check(
    gap_ivp < 1e-9,
    "and matches a high-order integrator on the trajectory",
    f"{gap_ivp:.0e} against DOP853 at rtol 1e-12: closed form and "
    "numerics, one flow",
)
validate.check(
    bool(np.all(eigs_h.real < 0)) and hump.max() > 2.0,
    "while a stable non-normal system triples before it decays",
    f"max ||e^At|| = {hump.max():.2f} with eigenvalues -1 and -1.2 — "
    "asymptotics belong to eigenvalues, transients to non-normality "
    "(3.5, in motion)",
)

## Exercise 4: Square roots, logarithms, and the round trips

**Part a)** On the SPD matrix of Exercise 1: gate
$(\sqrt{A})^2 = A$ at $10^{-12}$ and $e^{\log A} = A$ at $10^{-11}$
(`sqrtm`, `logm`, `expm`) — the functional calculus composing as the
scalar functions do, on the branch where all eigenvalues are
positive.

**Part b)** State the caveat by measurement: for a matrix with a
*negative* eigenvalue, `sqrtm` returns a complex result whose square
still reproduces $A$ (gate at $10^{-11}$) — the principal branch
does exactly what $\sqrt{-1}$ always did, and real inputs stop
implying real outputs the moment the spectrum crosses zero.

In [ ]:
# (solution hidden on the public site)


### Validation 4

In [ ]:
validate.check(
    rt_sqrt < 1e-12 and rt_log < 1e-11,
    "square root and logarithm round-trip on the SPD matrix",
    f"{rt_sqrt:.0e} and {rt_log:.0e}: the functional calculus composes "
    "like the scalar functions it lifts",
)
validate.check(
    rt_neg < 1e-11 and abs(R_neg[2, 2].imag - 3.0) < 1e-11,
    "and a negative eigenvalue sends the square root complex — "
    "correctly",
    "sqrt(-9) = 3i on the principal branch: real matrices stop having "
    "real functions the moment the spectrum crosses the branch cut",
)

## Exercise 5: Exponentials do not commute, at a measured rate

**Part a)** Gate the failure at $O(1)$: for random $4\times4$
$A, B$, $\lVert e^{A}e^{B} - e^{A+B}\rVert > 0.01$ — the scalar rule
$e^ae^b = e^{a+b}$ dies with commutativity (deterministic given the
seed; its exact size is reported).

**Part b)** Gate the *rate* of the failure {eq}`eq-mf-bch`: as
$s \to 0$, $\lVert e^{sA}e^{sB} - e^{s(A+B)}\rVert$ divided by
$\tfrac{s^2}{2}\lVert[A,B]\rVert$ approaches 1 — measured 1.04,
1.01, **1.003** at $s = 0.1, 0.03, 0.01$; gate within 10% at the
smallest scale, per the manifest. The commutator is the *first Lie bracket*,
and this ratio is where matrix groups begin — also where
split-operator ODE and quantum methods get their $O(s^2)$ error
budgets.

In [ ]:
# (solution hidden on the public site)


```{admonition} With your assistant
:class: tip
The BCH correction powers splitting methods. Ask your assistant for
`strang_split(A, B, t, n)` approximating $e^{t(A+B)}$ by $n$ steps of
the symmetric splitting $e^{\frac{t}{2n}A}e^{\frac{t}{n}B}
e^{\frac{t}{2n}A}$, then check it against the mathematics rather
than a demo: (i) the error falls as $n^{-2}$ (fit the exponent over
a decade of $n$, within 5%); (ii) the plain (Lie) splitting
$e^{\frac{t}{n}A}e^{\frac{t}{n}B}$ falls only as $n^{-1}$ — measure
both exponents side by side; (iii) when $[A, B] = 0$ (build $B$ as a
polynomial in $A$) both splittings are exact at every $n$, to
$10^{-12}$. The check is yours.
```

### Validation 5

In [ ]:
validate.check(
    big_gap > 0.01,
    "e^A e^B differs from e^(A+B) at order one (existence gated, size "
    "reported)",
    f"{big_gap:.2f} for the seeded pair — noncommutativity is not a "
    "small effect at scale one",
)
validate.check(
    abs(ratios[0.01] - 1.0) < 0.10,
    "and the failure's leading term is exactly the commutator (Eq. 3)",
    f"ratio {ratios[0.01]:.3f} at s = 0.01 (1.04 and 1.01 on the way "
    "down the scales) — BCH's s^2/2 [A,B], measured to 0.3%",
)

---
## Notebook summary

**The definition works where its hypotheses hold.** The spectral rule
matched `expm` and `sqrtm` at $10^{-13}$ on well-conditioned
eigenbases, and $\det e^{A} = e^{\operatorname{tr}A}$ held at
rounding — the exponential of anything is invertible.

**The dubious way failed on schedule, and the cure is
restructuring.** The Moler–Van Loan ladder lost seven digits, then
25%, then everything ($10^{22}$ relative) as intermediates reached
$10^{28}$ against $10^{-2}$ answers — cancellation as a measured
ledger. Hand-written scaling-and-squaring repaired all three rungs at
$10^{-14}$: the same series, tamed. Definition and algorithm are
different objects; this course's whole Volume V grew from that
sentence, and here is where Volume III proves it needs one.

**Dynamics behaved — including the misbehaviour.** The generator
property held by differences, the flow matched DOP853 at $10^{-10}$,
three phase portraits animated their taxonomy, and the stable
non-normal system climbed to 3.4 before decaying — eigenvalues own
the asymptote, non-normality owns the transient. Square roots and
logarithms round-tripped on the SPD branch and went honestly complex
across it.

**And exponentials multiply like matrices, not scalars.**
$e^{A}e^{B}$ missed $e^{A+B}$ by 28.5 at scale one, and the miss
tracked $\tfrac{s^2}{2}\lVert[A,B]\rVert$ to 0.3% as the scale
shrank — the commutator measured as the leading obstruction, which
is where Lie theory and splitting methods both begin.

**Methods introduced.** `f_eig` spectral evaluation,
cancellation-ledger analysis of series, hand-written
scaling-and-squaring, generator and integrator cross-checks,
transient-hump measurement, principal-branch round trips, and
BCH-rate measurement.

## Outlook

- **Padé, not Taylor.** Production `expm` pairs scaling-and-squaring
  with Padé approximants {cite}`higham2008functions` — fewer terms,
  same taming idea, plus backward-error control.
- **Krylov exponentials.** For the sparse matrices of
  [§5.3](../05-numerical/sparse-matrices.ipynb), $e^{At}b$ is
  computed in a Krylov subspace without ever forming $e^{At}$ —
  [§5.5](../05-numerical/krylov-gmres-preconditioning.ipynb)'s
  machinery pointed at dynamics.
- **The hump is a field.** Transient growth for stable systems is
  the core of non-modal stability theory (fluid transition,
  ecological resilience) — pseudospectra
  {cite}`trefethen2005spectra` quantify exactly how high the hump
  can go.
- **Exponentials on manifolds.** $e^{tA}$ for skew-symmetric $A$
  walks the rotation group — the doorway from matrix functions to
  Lie groups, with {eq}`eq-mf-bch` as the local map.

```{bibliography}
:filter: docname in docnames
```

In [ ]:
from ecp.style import footer

footer()